In [3]:
import os.path as osp
from modelscope.trainers import build_trainer
from modelscope.utils.hub import read_config

from modelscope.msdatasets import MsDataset
from datasets import concatenate_datasets


In [4]:
dataset_id = 'QBQTC'

# 载入训练集
train_dataset = MsDataset.load(dataset_id, namespace='damo', subset_name='default', split='train', keep_default_na=False)
dev_dataset = MsDataset.load(dataset_id, namespace='damo', subset_name='default', split='validation', keep_default_na=False)
train_dataset._hf_ds = concatenate_datasets([train_dataset._hf_ds, dev_dataset._hf_ds])  # 随版本更新该方法可能失效


2024-11-20 16:13:04,681 - modelscope - WARNING - Reusing dataset dataset_builder (/home/delta/.cache/modelscope/hub/datasets/damo/QBQTC/master/data_files)
2024-11-20 16:13:04,683 - modelscope - INFO - Generating dataset dataset_builder (/home/delta/.cache/modelscope/hub/datasets/damo/QBQTC/master/data_files)
2024-11-20 16:13:04,684 - modelscope - INFO - Loading meta-data file ...


0it [00:00, ?it/s]

100% 

2024-11-20 16:13:12,769 - modelscope - WARNING - Reusing dataset dataset_builder (/home/delta/.cache/modelscope/hub/datasets/damo/QBQTC/master/data_files)
2024-11-20 16:13:12,781 - modelscope - INFO - Generating dataset dataset_builder (/home/delta/.cache/modelscope/hub/datasets/damo/QBQTC/master/data_files)
2024-11-20 16:13:12,782 - modelscope - INFO - Loading meta-data file ...


0it [00:00, ?it/s]

In [5]:
# 载入公开测试集
eval_dataset = MsDataset.load(dataset_id, namespace='damo', subset_name='public', split='test', keep_default_na=False)
print("训练集：")
print(train_dataset._hf_ds)
print("公开测试集：")
print(eval_dataset._hf_ds)

2024-11-20 16:13:35,711 - modelscope - WARNING - Reusing dataset dataset_builder (/home/delta/.cache/modelscope/hub/datasets/damo/QBQTC/master/data_files)
2024-11-20 16:13:35,713 - modelscope - INFO - Generating dataset dataset_builder (/home/delta/.cache/modelscope/hub/datasets/damo/QBQTC/master/data_files)
2024-11-20 16:13:35,713 - modelscope - INFO - Loading meta-data file ...


0it [00:00, ?it/s]

训练集：
Dataset({
    features: ['id', 'query', 'title', 'label'],
    num_rows: 200000
})
公开测试集：
Dataset({
    features: ['id', 'query', 'title', 'label'],
    num_rows: 5000
})


In [6]:
model_id = 'damo/nlp_masts_backbone_clue_chinese-large'

WORK_DIR = './workspace'
BATCH_SIZE = 64  # 推荐使用官方的超参数

In [8]:
cfg = read_config(model_id, revision='v1.0.0')
cfg.train.work_dir = WORK_DIR
cfg_file = osp.join(WORK_DIR, 'train_config.json')
cfg.train.max_epochs = 7
# train_dataloader的配置
cfg.train.dataloader.batch_size_per_gpu = BATCH_SIZE
cfg.train.optimizer.lr = 2.0e-5
# lr_scheduler的配置
cfg.train.lr_scheduler = {
    'type': 'LinearLR',
    'start_factor': 1.0,
    'end_factor': 0.0,
    'total_iters':
    int(cfg.train.max_epochs * len(train_dataset) // BATCH_SIZE),
    'options': {
        'warmup': {
            'type': 'LinearWarmup',
            'warmup_iters': int(cfg.train.max_epochs * len(train_dataset) * 0.9 // BATCH_SIZE)
        },
        'by_epoch': False
    }
}
cfg.dump(cfg_file)

2024-11-20 16:14:28,326 - modelscope - INFO - Use user-specified model revision: v1.0.0


In [9]:
kwargs = dict(
    model=model_id,
    model_revision='v1.0.0',
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    cfg_file=cfg_file,
)
trainer = build_trainer(default_args=kwargs)

2024-11-20 16:14:45,151 - modelscope - INFO - Use user-specified model revision: v1.0.0


2024-11-20 16:14:46,020 - modelscope - INFO - Use user-specified model revision: v1.0.0


2024-11-20 16:15:09,333 - modelscope - INFO - Use user-specified model revision: v1.0.0
2024-11-20 16:15:09,595 - modelscope - INFO - initialize model from /home/delta/.cache/modelscope/hub/damo/nlp_masts_backbone_clue_chinese-large
2024-11-20 16:15:13,757 - modelscope - INFO - head has no _keys_to_ignore_on_load_missing
/home/delta/anaconda3/envs/modelscope/lib/python3.8/site-packages/modelscope/utils/checkpoint.py:550: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unles

In [10]:
print('===============================================================')
print('pre-trained model loaded, training started:')
print('===============================================================')

trainer.train()

print('===============================================================')
print('train success.')
print('===============================================================')

for i in range(cfg.train.max_epochs):
    eval_results = trainer.evaluate(f'{WORK_DIR}/epoch_{i+1}.pth')
    print(f'epoch {i} evaluation result:')
    print(eval_results)

    
print('===============================================================')
print('evaluate success')
print('===============================================================')

2024-11-20 16:16:42,374 - modelscope - WARNING - ('OPTIMIZER', 'default', 'AdamW') not found in ast index file
2024-11-20 16:16:42,378 - modelscope - WARNING - ('LR_SCHEDULER', 'default', 'LinearLR') not found in ast index file
2024-11-20 16:16:42,385 - modelscope - INFO - Stage: before_run:
    (ABOVE_NORMAL) OptimizerHook                      
    (LOW         ) LrSchedulerHook                    
    (LOW         ) CheckpointHook                     
    (VERY_LOW    ) TextLoggerHook                     
 -------------------- 
Stage: before_train_epoch:
    (LOW         ) LrSchedulerHook                    
 -------------------- 
Stage: before_train_iter:
    (ABOVE_NORMAL) OptimizerHook                      
 -------------------- 
Stage: after_train_iter:
    (ABOVE_NORMAL) OptimizerHook                      
    (NORMAL      ) EvaluationHook                     
    (LOW         ) LrSchedulerHook                    
    (LOW         ) CheckpointHook                     
    (VERY_

pre-trained model loaded, training started:


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
/home/delta/anaconda3/envs/modelscope/lib/python3.8/site-packages/transformers/modeling_utils.py:1161: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


: 